# Full Drug Discovery Pipeline: From Screening to Simulation

This notebook demonstrates running the complete epistemic drug discovery pipeline:
1. **Molecular screening** (Lipinski's Rule of Five)
2. **Pharmacokinetic fitting** (1-compartment model)
3. **Virtual population simulation** (efficacy & safety)
4. **Report generation** with uncertainty propagation

In [ ]:
import sys
sys.path.insert(0, '../sounio-py/python')

from sounio.knowledge import Knowledge
from sounio.types import (
    Molecule,
    PKParameters,
    PatientData,
    SimulationResult,
    PipelineResult,
    ScreeningResult,
)
from sounio.report import ReportBuilder

print("Sounio modules loaded successfully")

## Part 1: Molecular Screening

Filter compounds by Lipinski's Rule of Five (Ro5):
- MW ≤ 500 Da
- logP ≤ 5
- HBD ≤ 5
- HBA ≤ 10

In [ ]:
# Create a library of candidate molecules
library = [
    Molecule(
        name="Compound_A",
        smiles="CC(=O)Oc1ccccc1C(=O)O",
        molecular_weight=Knowledge(180.2, 0.5, "chemdb"),
        logp=Knowledge(1.2, 0.3, "ADME_pred"),
        hbd=2,
        hba=4,
    ),
    Molecule(
        name="Compound_B",
        smiles="c1ccc(cc1)c2ccccc2",
        molecular_weight=Knowledge(154.2, 0.4, "chemdb"),
        logp=Knowledge(4.1, 0.5, "ADME_pred"),
        hbd=0,
        hba=0,
    ),
    Molecule(
        name="Compound_C",
        smiles="NC(=O)C1=CC=C(C=C1)O",
        molecular_weight=Knowledge(137.1, 0.4, "chemdb"),
        logp=Knowledge(0.5, 0.2, "ADME_pred"),
        hbd=3,
        hba=3,
    ),
    Molecule(
        name="Compound_D",
        smiles="CC(C)(C)c1ccc(O)cc1",
        molecular_weight=Knowledge(150.2, 0.5, "chemdb"),
        logp=Knowledge(3.5, 0.4, "ADME_pred"),
        hbd=1,
        hba=1,
    ),
]

print(f"Screening library: {len(library)} compounds\n")

# Apply Ro5 filter
passed = []
failed = []

for mol in library:
    ro5_ok = mol.lipinski_rule_of_five()
    if ro5_ok:
        passed.append(mol)
        print(f"✓ {mol.name:15s} PASS (MW={mol.molecular_weight.value:.1f}, logP={mol.logp.value:.2f})")
    else:
        failed.append(mol)
        print(f"✗ {mol.name:15s} FAIL (MW={mol.molecular_weight.value:.1f}, logP={mol.logp.value:.2f})")

print(f"\nScreening results: {len(passed)}/{len(library)} passed")

## Part 2: Pharmacokinetic Parameter Fitting

Fit 1-compartment PK model from clinical data.

In [ ]:
# Simulate PK fitting results (in practice, these would come from parameter estimation)
pk_results = {
    "Compound_A": PKParameters(
        clearance=Knowledge(15.2, 2.1, "noncompartmental_analysis"),
        volume_dist=Knowledge(72.0, 5.0, "fitting"),
        half_life=Knowledge(3.35, 0.40, "derived_ln2_vd_cl"),
        bioavailability=Knowledge(0.82, 0.06, "absolute_ba_study"),
    ),
    "Compound_B": PKParameters(
        clearance=Knowledge(8.5, 1.2, "noncompartmental_analysis"),
        volume_dist=Knowledge(95.0, 8.0, "fitting"),
        half_life=Knowledge(7.70, 0.95, "derived_ln2_vd_cl"),
        bioavailability=Knowledge(0.45, 0.08, "absolute_ba_study"),
    ),
    "Compound_C": PKParameters(
        clearance=Knowledge(22.0, 3.5, "noncompartmental_analysis"),
        volume_dist=Knowledge(58.0, 4.5, "fitting"),
        half_life=Knowledge(1.83, 0.28, "derived_ln2_vd_cl"),
        bioavailability=Knowledge(0.91, 0.05, "absolute_ba_study"),
    ),
    "Compound_D": PKParameters(
        clearance=Knowledge(12.0, 2.0, "noncompartmental_analysis"),
        volume_dist=Knowledge(80.0, 6.0, "fitting"),
        half_life=Knowledge(4.62, 0.75, "derived_ln2_vd_cl"),
        bioavailability=Knowledge(0.65, 0.07, "absolute_ba_study"),
    ),
}

print("Pharmacokinetic Parameters (passed compounds):\n")
for mol in passed:
    pk = pk_results[mol.name]
    print(f"{mol.name}:")
    print(f"  Clearance:   {pk.clearance.value:.2f} ± {pk.clearance.epsilon:.2f} L/h  ({pk.clearance.relative_uncertainty:.1%} rel unc)")
    print(f"  Volume Dist: {pk.volume_dist.value:.1f} ± {pk.volume_dist.epsilon:.1f} L    ({pk.volume_dist.relative_uncertainty:.1%} rel unc)")
    print(f"  Half-life:   {pk.half_life.value:.2f} ± {pk.half_life.epsilon:.2f} h    ({pk.half_life.relative_uncertainty:.1%} rel unc)")
    print(f"  Bioavail:    {pk.bioavailability.value:.2f} ± {pk.bioavailability.epsilon:.2f}       ({pk.bioavailability.relative_uncertainty:.1%} rel unc)")
    print()

## Part 3: Virtual Population Simulation

Simulate efficacy and safety outcomes in a virtual patient population.

In [ ]:
# Simulate population-level outcomes for each compound
simulation_results = {}

for mol in passed:
    pk = pk_results[mol.name]
    
    # Estimate efficacy based on PK parameters
    # (In reality, these come from PK/PD models)
    efficacy = (pk.bioavailability * 0.95) * Knowledge(1.0, 0.08, "pk_pd_link")
    efficacy = Knowledge(
        max(0.0, min(1.0, efficacy.value)),
        efficacy.epsilon,
        "population_simulation"
    )
    
    # Adverse events inversely related to half-life (longer = safer)
    adverse = Knowledge(0.20, 0.04, "population_simulation") / (pk.half_life / Knowledge(5.0, 0.5, "reference"))
    adverse = Knowledge(
        max(0.0, min(1.0, adverse.value)),
        adverse.epsilon,
        "population_simulation"
    )
    
    # Therapeutic index
    ti = efficacy / (adverse + Knowledge(0.01, 0.001, "baseline"))
    
    simulation_results[mol.name] = SimulationResult(
        efficacy_rate=efficacy,
        adverse_rate=adverse,
        therapeutic_index=ti,
        confidence=min(1.0, efficacy.confidence * 0.9),
        n_patients=5000,
    )

print("Population Simulation Results (N=5000 virtual patients):\n")
for mol in passed:
    sim = simulation_results[mol.name]
    print(f"{mol.name}:")
    print(f"  Efficacy:            {sim.efficacy_rate.value:.3f} ± {sim.efficacy_rate.epsilon:.3f}")
    print(f"  Adverse rate:        {sim.adverse_rate.value:.3f} ± {sim.adverse_rate.epsilon:.3f}")
    print(f"  Therapeutic index:   {sim.therapeutic_index.value:.2f}")
    print(f"  Confidence:          {sim.confidence:.1%}")
    print()

## Part 4: Pipeline Summary

In [ ]:
# Select best compound (highest therapeutic index)
best_compound = max(
    [(mol, simulation_results[mol.name]) for mol in passed],
    key=lambda x: x[1].therapeutic_index.value
)

result = PipelineResult(
    molecules_screened=len(library),
    molecules_passed=len(passed),
    pk_fitted=len(passed),
    simulation=best_compound[1],
    provenance_chain=[
        "molecular_screening_lipinski",
        "pk_fitting_nca",
        "population_simulation_5k",
        f"best_compound_{best_compound[0].name}",
    ],
    exit_code=0,
)

print("Pipeline Execution Summary:")
print(result.summary())

## Part 5: Generate Report

Create a reproducible PDF/Markdown report with all results.

In [ ]:
# Create report
rb = ReportBuilder(
    "Drug Discovery Pipeline Report",
    author="Epistemic Computing Team",
    date="2026-03-18",
)

rb.add_section(
    "Executive Summary",
    f"""This report presents the results of an epistemic drug discovery pipeline.
    
**Screening Phase**: {result.molecules_screened} molecules evaluated, {result.molecules_passed} passed Lipinski's Rule of Five.

**Pharmacokinetic Phase**: All passing compounds underwent 1-compartment PK parameter estimation with uncertainty quantification via GUM.

**Simulation Phase**: Virtual population simulation (N={result.simulation.n_patients}) evaluated efficacy and safety outcomes.

**Recommendation**: {best_compound[0].name} selected as lead compound based on therapeutic index.
"""
)

rb.add_section(
    "Methods",
    """**Screening**: Lipinski's Rule of Five criteria applied to molecular properties (MW ≤500, logP ≤5, HBD ≤5, HBA ≤10).

**PK Fitting**: 1-compartment open model with first-order elimination fit by noncompartmental analysis (NCA). Uncertainty propagated via GUM.

**Simulation**: Virtual population generated from PK/PD relationships; efficacy estimated from bioavailability, adverse events from half-life.
"""
)

# Screening summary
screening_table = {mol.name: Knowledge(1.0, 0.0, "passed") for mol in passed}
rb.add_knowledge_table("Compounds Passing Screening", screening_table)

# PK parameters for lead compound
best_pk = pk_results[best_compound[0].name]
rb.add_knowledge_table(
    f"Lead Compound ({best_compound[0].name}) Pharmacokinetic Parameters",
    {
        "Clearance (L/h)": best_pk.clearance,
        "Volume of Distribution (L)": best_pk.volume_dist,
        "Half-life (h)": best_pk.half_life,
        "Bioavailability": best_pk.bioavailability,
    },
)

rb.add_pipeline_summary(result)

print("\n" + "="*60)
print("MARKDOWN REPORT PREVIEW")
print("="*60 + "\n")
md_report = rb.to_markdown()
print(md_report[:1500] + "\n...\n[Report continues]")

# Save report
rb.save("drug_discovery_report.md", format="markdown")
print(f"\nReport saved to: drug_discovery_report.md")

## Part 6: Access Generated Report

In [ ]:
# Read and display saved report
with open("drug_discovery_report.md", "r") as f:
    report_content = f.read()

print("Generated Report (first 2000 characters):")
print("\n" + "="*60)
print(report_content[:2000])
print("\n[... report continues ...]")
print("="*60)

## Summary

This notebook demonstrated:
1. **Molecular screening** with Lipinski's Rule of Five
2. **PK parameter fitting** with uncertainty quantification
3. **Population simulation** with epistemic aggregation
4. **Report generation** preserving provenance and uncertainty

Next: See notebook 04 for clinical data integration (SDTM).